# Aligning timestamps to label window

### Aim

The goal here is to document our progress in making a function which aligns article publication timestamps to our label windows, as well as note any edge cases which we need to care for and conventions we are using. We also want to save the schedule we will be using for the rest of the project so we can use it when calculating our features and labels.

**Output:** `schedule.parquet`

### Importance

The code for this function is important to the project as it is the leakage frontier: if an article published *after* a price move gets labeled with the move it's reporting on, every downstream result becomes impressive-looking fiction, and no error will ever be thrown. We need to make sure that we are following strict conventions and accounting for any holidays or unusual market closures.

### Convention

Let's document our conventions which we will use for the function. Essentially there are two possible timestamps we can come across: *market hours* and *after market hours*. Since we are working around the NYSE, we need to make sure the timestamps correspond to the correct timezone. We are working in **UTC** so we must make sure that the label windows correspond with this.

A regular trading day in the NYSE opens at 09:30 ET and closes at 16:00 ET , and on a regular week it is open Monday to Friday. Since all of our timestamps are in UTC, we need to make sure the times match this timezone. Because ET change their clocks twice a year, the regular market times in UTC change based on the time of the year:
- March to October -> 13:30 UTC to 20:00 UTC
- October to March -> 14:30 UTC to 21:00 UTC

The bottom line here is: Every timestamp should correspond to the **next full trading day** so we can start our label window from there. Let's lay out some examples (Assuming we are in August).
 
- During market hours -> next market opening
    - E.g. Friday 19:00 UTC -> Monday 13:30 UTC 
    - E.g. Wednesday 19:59 UTC -> Thursday 13:30 UTC
- After market hours -> next market opening
    - E.g. Sunday 13:00 UTC -> Monday 13:30 UTC
    - E.g. Tuesday 13:29 UTC -> Tuesday 13:30 UTC
- Abnormal holidays -> next market opening
    - E.g. 1st January 12:00 UTC (New Years Day Closure) -> 2nd January 14:30 UTC

### Pulling Market Calendar

To begin with, let's pull the real market calendar data we will be using for this function using `pandas_market_calendars`. Similarly to when we were pulling OHLCV data, we need to make sure that we are pulling a wider range than our publication date range, since we need to be able to look forward 3 full trading days for our abnormal return label. 

For this purpose it's enough to look forward by 10 days sice this accounts for any weekends and holidays as well as giving an extra few day window afterwards. We also extend the timeframe at the lower end by just one day to account for articles published the morning of the beginning of our timeframe before trading begins, since we still want these to be accepted.

In [15]:
import pandas_market_calendars as mcal
import pandas as pd
import datetime as dt
from stock_predictor.config import NEWS_START_DATE, NEWS_END_DATE, RAW_DATA_DIR

schedule_start = pd.Timestamp(NEWS_START_DATE) - dt.timedelta(days=1)
schedule_end = pd.Timestamp(NEWS_END_DATE) + dt.timedelta(days=10)

print(f"Schedule start date: {schedule_start}")
print(f"Schedule end date: {schedule_end}")

nyse = mcal.get_calendar("NYSE")

schedule = nyse.schedule(start_date=schedule_start, end_date=schedule_end)
print(schedule)
print(schedule.dtypes)

Schedule start date: 2025-07-31 00:00:00
Schedule end date: 2026-08-11 00:00:00
                         market_open              market_close
2025-07-31 2025-07-31 13:30:00+00:00 2025-07-31 20:00:00+00:00
2025-08-01 2025-08-01 13:30:00+00:00 2025-08-01 20:00:00+00:00
2025-08-04 2025-08-04 13:30:00+00:00 2025-08-04 20:00:00+00:00
2025-08-05 2025-08-05 13:30:00+00:00 2025-08-05 20:00:00+00:00
2025-08-06 2025-08-06 13:30:00+00:00 2025-08-06 20:00:00+00:00
...                              ...                       ...
2026-08-05 2026-08-05 13:30:00+00:00 2026-08-05 20:00:00+00:00
2026-08-06 2026-08-06 13:30:00+00:00 2026-08-06 20:00:00+00:00
2026-08-07 2026-08-07 13:30:00+00:00 2026-08-07 20:00:00+00:00
2026-08-10 2026-08-10 13:30:00+00:00 2026-08-10 20:00:00+00:00
2026-08-11 2026-08-11 13:30:00+00:00 2026-08-11 20:00:00+00:00

[259 rows x 2 columns]
market_open     datetime64[us, UTC]
market_close    datetime64[us, UTC]
dtype: object


We can see that we have datetimes in UTC which is what we need. This can now be saved as a parquet file so it is ready for future use.

In [16]:
schedule.to_parquet(RAW_DATA_DIR / "raw_schedule.parquet")

Let's check for holidays so we can use them as an edge case when testing the function. We can do this by checking for every weekday which is not included in our list of trading days.

In [17]:
# Get all weekdays
all_days = pd.date_range(start=NEWS_START_DATE, end=schedule_end, freq="D")
weekdays = all_days[all_days.dayofweek < 5]

# Normalise trading days
trading_days = schedule.index.normalize()

# Filter to show holidays i.e. non trading days
holidays = weekdays[~weekdays.normalize().isin(trading_days)].normalize()

print(holidays)

DatetimeIndex(['2025-09-01', '2025-11-27', '2025-12-25', '2026-01-01',
               '2026-01-19', '2026-02-16', '2026-04-03', '2026-05-25',
               '2026-06-19', '2026-07-03'],
              dtype='datetime64[us]', freq=None)


We can now use this information to write our function and tests.

### Showcasing examples

Let's now apply our function to the examples we gave in our convention to show it works. We can write a function which makes timezone aware datetime objects which allow us to easily pass datetimes in as they would be from the article timestamps. They also need to be timezone aware so that pandas is able to make comparisons between the timestamps.

In [18]:
from stock_predictor.market.timestamp_alignment import align_timestamp, make_dt

example_cases = [
    {"label": "Friday market hours",     "date": "2026-07-03", "time": "19:00:00"},
    {"label": "Wednesday just before close", "date": "2026-07-01", "time": "19:59:00"},
    {"label": "Sunday after-hours",      "date": "2026-07-05", "time": "13:00:00"},
    {"label": "Tuesday just before open","date": "2026-07-07", "time": "13:29:00"},
    {"label": "Christmas Day",           "date": "2026-01-01", "time": "12:00:00"},
]

for case in example_cases:
    pub_dt = make_dt(case["date"], case["time"])
    result = align_timestamp(pub_dt, schedule)
    print(f"{case['label']:35s} | {pub_dt} -> {result}")

Friday market hours                 | 2026-07-03 19:00:00+00:00 -> 2026-07-06 13:30:00+00:00
Wednesday just before close         | 2026-07-01 19:59:00+00:00 -> 2026-07-02 13:30:00+00:00
Sunday after-hours                  | 2026-07-05 13:00:00+00:00 -> 2026-07-06 13:30:00+00:00
Tuesday just before open            | 2026-07-07 13:29:00+00:00 -> 2026-07-07 13:30:00+00:00
Christmas Day                       | 2026-01-01 12:00:00+00:00 -> 2026-01-02 14:30:00+00:00


We see the function behaves as intended in our examples. This will also be tested more formally using thorough unit tests which will test all edge cases.

### Note

It should be noted that the function takes in the NYSE schedule as an argument as well as the timestamp of the publication. This is so that we save having to pull a new schedule for each function call if we know that all of our articles will sit within our timeframe. We can pull the schedule once and use the same one for each timestamp.

Also, the function checks that the publication timstamp is within the timeframe of the schedule. This works because, similarly to earlier in the notebook, we pull a schedule based on the timeframe the articles are pulled from (and even add a leeway to this), so no article should fall outside the schedule. There is also a separate check for making sure that the publication date itself is within our timeframe, which makes sure that this raises an error even if the schedule is correct.

As mentioned earlier, we make sure that articles published in the morning of the start of our timeframe are still accepted and the label window begins on the same day.